# Lilly — train the Bosnian to English model

Run the cells top to bottom (Runtime -> Run all also works).

**Before you start:** Runtime -> Change runtime type -> select **T4 GPU** -> Save.

Total time: roughly 2-3 hours on a free T4. Keep the tab open.

In [ ]:
# 1. Check we actually got a GPU
!nvidia-smi -L

In [ ]:
# 2. Get the Lilly code
!git clone https://github.com/ssaaffaakk/Lilly.git
%cd Lilly

In [ ]:
# 3. Install what we need (~2 min) — same versions as on our Mac
!pip -q install transformers==4.49.0 peft==0.14.0 accelerate==1.3.0 sacrebleu sentencepiece sacremoses

In [ ]:
# 4. Point Colab at Lilly's model folder
# The weights are not in the repo (too big for git). Upload your local
# models/lilly/translate folder to Google Drive once, then this cell finds it.
import os
from google.colab import drive
drive.mount('/content/drive')
os.environ['LILLY_BASE'] = '/content/drive/MyDrive/lilly/translate'
print('base weights:', os.environ['LILLY_BASE'],
      '->', os.path.isdir(os.environ['LILLY_BASE']))


In [ ]:
# 5. Download and clean the Bosnian-English data (~5 min)
!python3 data/scripts/download_data.py
!python3 data/scripts/clean_data.py

In [ ]:
# 6. Quick pipeline check (~3 min) — tiny run just to prove everything works
!python3 training/train_translation.py --quick-test

In [ ]:
# 7. THE REAL TRAINING (~2-3 hours)
!python3 training/train_translation.py

In [ ]:
# 8. Score it: base model vs our Lilly on sentences it never saw (~20 min)
!python3 training/evaluate.py --adapter models/lilly/adapter --limit 500
!cat training/RESULTS.md

In [ ]:
# 9. Download the trained model (small file, ~20-40 MB)
!zip -qr lilly-adapter.zip models/lilly/adapter training/RESULTS.md
from google.colab import files
files.download('lilly-adapter.zip')

**Done!** Send the downloaded `lilly-adapter.zip` back into the project — unzip it so the
adapter sits at `models/lilly/adapter/`. That little file is our trained model, and the app
picks it up automatically the next time it starts.
